# 09 Merge OncoKB HGVSg and HGVSp outputs by exact variant key

This notebook merges paired OncoKB output files from the same MAF input.

For each sample, it:

- merges HGVSg and HGVSp outputs by the real variant/caller key: `Chr`, `Start`, `End`, `Ref`, `Alt`, `caller`;
- does **not** use row order or an artificial row index for the merge;
- saves an all-rows merged file;
- saves a duplicate-variant audit file;
- saves a final one-row-per-exact-variant file with Mutect2 and DeepVariant evidence columns placed immediately after `Chr`, `Start`, `End`, `Ref`, `Alt`.

The final wide file makes it easy to see whether a variant was found by Mutect2, DeepVariant, or both callers.

In [ ]:
from pathlib import Path
import pandas as pd

base_dir =  Path(r"/project/knathans_shared/donetski/Notebooks/OutputFiles")

sg_dir = base_dir / "split_oncokb_hgvsg_outputs"
sp_dir = base_dir / "split_oncokb_hgvsp_outputs"
output_dir = base_dir / "09_merged_HGVSg_HGVSp_by_exact_variant"

samples_to_process = "all"
# options:
# "all"
# "1703-DB1A"
# ["1703-DB1A", "another-sample"]

sg_suffix = "_oncokb_hgvsg_output.csv"
sp_suffix = "_oncokb_hgvsp_output.csv"
output_prefix = "09"

variant_columns = ["Chr", "Start", "End", "Ref", "Alt"]
caller_column = "caller"
merge_key = [*variant_columns, caller_column]

output_dir.mkdir(parents=True, exist_ok=True)

## Helper functions

In [ ]:
def resolve_samples(samples_to_process):
    if isinstance(samples_to_process, str) and samples_to_process.lower() == "all":
        sg_files = sorted(sg_dir.glob(f"*{sg_suffix}"))
        if not sg_files:
            raise FileNotFoundError(f"No HGVSg files found in {sg_dir}")
        return [path.name.removesuffix(sg_suffix) for path in sg_files]

    if isinstance(samples_to_process, str):
        return [samples_to_process]

    return list(samples_to_process)


def require_columns(df, columns, label):
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"{label} missing required columns: {missing}")


def clean_key_columns(df, columns):
    for column in columns:
        df[column] = df[column].astype(str).str.strip()


def combine_unique_values(series):
    values = []
    for value in series:
        if pd.isna(value):
            continue
        value = str(value).strip()
        if value and value.lower() != "nan" and value not in values:
            values.append(value)
    return " | ".join(values)


def coalesce_columns(df, columns):
    result = pd.Series("", index=df.index, dtype="object")

    for column in columns:
        if column not in df.columns:
            continue
        values = df[column].astype(str).str.strip().replace("nan", "")
        result = result.mask(result.eq("") & values.ne(""), values)

    return result


def save_duplicate_key_audit(df, key_columns, output_path, label):
    duplicate_mask = df.duplicated(subset=key_columns, keep=False)
    duplicate_df = df.loc[duplicate_mask].sort_values(key_columns).reset_index(drop=True)

    if not duplicate_df.empty:
        duplicate_df.to_csv(output_path, index=False, encoding="utf-8-sig")
        display(duplicate_df[key_columns].head(50))
        raise ValueError(
            f"{label} has duplicate merge keys. "
            f"Audit saved to: {output_path}"
        )


def caller_label(series):
    return series.astype(str).str.strip().str.lower().map(
        {
            "mutect2": "Mutect2",
            "deepvariant": "DeepVariant",
        }
    )

## Caller-specific depth/VAF columns

The final file is one row per exact variant. These helper functions create the Mutect2 and DeepVariant columns from the caller-specific rows.

In [ ]:
def make_caller_evidence_table(all_rows_df):
    work_df = all_rows_df.copy()
    work_df["_caller_label"] = caller_label(work_df[caller_column])

    unknown_callers = sorted(
        work_df.loc[work_df["_caller_label"].isna(), caller_column]
        .astype(str)
        .unique()
    )
    if unknown_callers:
        raise ValueError(f"Unexpected caller values: {unknown_callers}")

    work_df["_total_depth"] = coalesce_columns(
        work_df,
        ["Sample.Depth_HGVSg", "Sample.Depth_HGVSp", "Sample.Depth"],
    )
    work_df["_alt_depth"] = coalesce_columns(
        work_df,
        ["Sample.AltDepth_HGVSg", "Sample.AltDepth_HGVSp", "Sample.AltDepth"],
    )
    work_df["_vaf"] = coalesce_columns(
        work_df,
        ["Sample.AltFrac_HGVSg", "Sample.AltFrac_HGVSp", "Sample.AltFrac"],
    )

    final_df = work_df[variant_columns].drop_duplicates().reset_index(drop=True)

    for caller in ["Mutect2", "DeepVariant"]:
        caller_df = work_df.loc[work_df["_caller_label"].eq(caller)]

        caller_summary = (
            caller_df
            .groupby(variant_columns, as_index=False, sort=False, dropna=False)
            .agg(
                {
                    "_total_depth": combine_unique_values,
                    "_alt_depth": combine_unique_values,
                    "_vaf": combine_unique_values,
                }
            )
            .rename(
                columns={
                    "_total_depth": f"{caller}_total_depth",
                    "_alt_depth": f"{caller}_alt_depth",
                    "_vaf": f"{caller}_VAF",
                }
            )
        )
        caller_summary[f"{caller}_found"] = "Y"

        final_df = final_df.merge(caller_summary, on=variant_columns, how="left")
        final_df[f"{caller}_found"] = final_df[f"{caller}_found"].fillna("N")

        for column in [f"{caller}_total_depth", f"{caller}_alt_depth", f"{caller}_VAF"]:
            final_df[column] = final_df[column].fillna("")

    final_df["Callers"] = final_df.apply(
        lambda row: " | ".join(
            caller
            for caller in ["Mutect2", "DeepVariant"]
            if row[f"{caller}_found"] == "Y"
        ),
        axis=1,
    )

    first_columns = [
        *variant_columns,
        "Mutect2_found",
        "Mutect2_total_depth",
        "Mutect2_alt_depth",
        "Mutect2_VAF",
        "DeepVariant_found",
        "DeepVariant_total_depth",
        "DeepVariant_alt_depth",
        "DeepVariant_VAF",
        "Callers",
    ]

    return final_df[first_columns]


def make_final_variant_table(all_rows_df):
    caller_evidence_df = make_caller_evidence_table(all_rows_df)

    raw_caller_columns = [
        caller_column,
        "Sample.Depth", "Sample.Depth_HGVSg", "Sample.Depth_HGVSp",
        "Sample.AltDepth", "Sample.AltDepth_HGVSg", "Sample.AltDepth_HGVSp",
        "Sample.AltFrac", "Sample.AltFrac_HGVSg", "Sample.AltFrac_HGVSp",
    ]

    annotation_columns = [
        column
        for column in all_rows_df.columns
        if column not in [*variant_columns, *raw_caller_columns]
    ]

    annotation_df = (
        all_rows_df
        .groupby(variant_columns, as_index=False, sort=False, dropna=False)
        .agg({column: combine_unique_values for column in annotation_columns})
    )

    final_df = caller_evidence_df.merge(
        annotation_df,
        on=variant_columns,
        how="left",
        validate="one_to_one",
    )

    first_columns = list(caller_evidence_df.columns)
    remaining_columns = [column for column in final_df.columns if column not in first_columns]
    return final_df[first_columns + remaining_columns]

## Merge one sample

HGVSg and HGVSp are merged by `Chr`, `Start`, `End`, `Ref`, `Alt`, and `caller`. The final table then collapses to one row per exact genomic variant and widens Mutect2/DeepVariant evidence into separate columns.

In [ ]:
def merge_one_sample(sample_id):
    sg_path = sg_dir / f"{sample_id}{sg_suffix}"
    sp_path = sp_dir / f"{sample_id}{sp_suffix}"
    sample_output_dir = output_dir / sample_id
    sample_output_dir.mkdir(parents=True, exist_ok=True)

    if not sg_path.exists():
        raise FileNotFoundError(f"Missing HGVSg file: {sg_path}")
    if not sp_path.exists():
        raise FileNotFoundError(f"Missing HGVSp file: {sp_path}")

    sg_df = pd.read_csv(sg_path, dtype=str, keep_default_na=False).reset_index(drop=True)
    sp_df = pd.read_csv(sp_path, dtype=str, keep_default_na=False).reset_index(drop=True)

    require_columns(sg_df, merge_key, "HGVSg")
    require_columns(sp_df, merge_key, "HGVSp")
    clean_key_columns(sg_df, merge_key)
    clean_key_columns(sp_df, merge_key)

    save_duplicate_key_audit(
        sg_df,
        merge_key,
        sample_output_dir / f"{output_prefix}_{sample_id}_duplicate_HGVSg_merge_keys.csv",
        "HGVSg",
    )
    save_duplicate_key_audit(
        sp_df,
        merge_key,
        sample_output_dir / f"{output_prefix}_{sample_id}_duplicate_HGVSp_merge_keys.csv",
        "HGVSp",
    )

    sg_df = sg_df.rename(
        columns={column: f"{column}_HGVSg" for column in sg_df.columns if column not in merge_key}
    )
    sp_df = sp_df.rename(
        columns={column: f"{column}_HGVSp" for column in sp_df.columns if column not in merge_key}
    )

    all_rows_df = sg_df.merge(
        sp_df,
        on=merge_key,
        how="outer",
        indicator=True,
        validate="one_to_one",
    )

    unmatched_df = all_rows_df.loc[all_rows_df["_merge"].ne("both")].copy()
    if not unmatched_df.empty:
        unmatched_path = sample_output_dir / f"{output_prefix}_{sample_id}_unmatched_HGVSg_HGVSp_rows.csv"
        unmatched_df.to_csv(unmatched_path, index=False, encoding="utf-8-sig")
        display(unmatched_df[[*merge_key, "_merge"]].head(50))
        raise ValueError(
            f"{sample_id}: HGVSg and HGVSp do not contain the same variant/caller keys. "
            f"Unmatched rows saved to: {unmatched_path}"
        )

    all_rows_df = all_rows_df.drop(columns="_merge")
    all_rows_df.insert(0, "SOURCE_SAMPLE", sample_id)

    ordered_start = ["SOURCE_SAMPLE", *merge_key]
    remaining_columns = [column for column in all_rows_df.columns if column not in ordered_start]
    all_rows_df = all_rows_df[ordered_start + remaining_columns]

    duplicate_mask = all_rows_df.duplicated(subset=variant_columns, keep=False)
    duplicate_rows_df = (
        all_rows_df.loc[duplicate_mask]
        .sort_values([*variant_columns, caller_column])
        .reset_index(drop=True)
    )

    final_df = make_final_variant_table(all_rows_df)

    all_rows_output_path = sample_output_dir / f"{output_prefix}_{sample_id}_combined_HGVSg_HGVSp_all_rows.csv"
    duplicate_audit_path = sample_output_dir / f"{output_prefix}_{sample_id}_exact_duplicate_rows_audit.csv"
    final_output_path = sample_output_dir / f"{output_prefix}_{sample_id}_combined_HGVSg_HGVSp_unique_exact_variants.csv"

    all_rows_df.to_csv(all_rows_output_path, index=False, encoding="utf-8-sig")
    duplicate_rows_df.to_csv(duplicate_audit_path, index=False, encoding="utf-8-sig")
    final_df.to_csv(final_output_path, index=False, encoding="utf-8-sig")

    mutect2_found = final_df["Mutect2_found"].eq("Y")
    deepvariant_found = final_df["DeepVariant_found"].eq("Y")

    return {
        "sample_id": sample_id,
        "hgvsg_rows": len(sg_df),
        "hgvsp_rows": len(sp_df),
        "all_merged_rows": len(all_rows_df),
        "unique_exact_variants": len(final_df),
        "mutect2_only_variants": int((mutect2_found & ~deepvariant_found).sum()),
        "deepvariant_only_variants": int((deepvariant_found & ~mutect2_found).sum()),
        "both_callers_variants": int((mutect2_found & deepvariant_found).sum()),
        "duplicate_audit_rows": len(duplicate_rows_df),
        "all_rows_output": str(all_rows_output_path),
        "duplicate_audit_output": str(duplicate_audit_path),
        "unique_exact_variants_output": str(final_output_path),
    }

## Run the merge

In [ ]:
samples = resolve_samples(samples_to_process)
print(f"Samples to process: {len(samples):,}")

summary_rows = []
for sample_id in samples:
    print(f"\nProcessing {sample_id}")
    summary = merge_one_sample(sample_id)
    summary_rows.append(summary)
    print(f"  all rows: {summary['all_merged_rows']:,}")
    print(f"  unique exact variants: {summary['unique_exact_variants']:,}")
    print(f"  Mutect2 only: {summary['mutect2_only_variants']:,}")
    print(f"  DeepVariant only: {summary['deepvariant_only_variants']:,}")
    print(f"  both callers: {summary['both_callers_variants']:,}")

summary_df = pd.DataFrame(summary_rows)
summary_path = output_dir / f"{output_prefix}_HGVSg_HGVSp_merge_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nSaved summary:", summary_path)
summary_df

## Output files

For each sample, the notebook saves:

- `*_combined_HGVSg_HGVSp_all_rows.csv`: all HGVSg/HGVSp rows merged by exact variant + caller;
- `*_exact_duplicate_rows_audit.csv`: variants present in more than one caller row, usually Mutect2 and DeepVariant;
- `*_combined_HGVSg_HGVSp_unique_exact_variants.csv`: one row per exact variant, with Mutect2/DeepVariant evidence columns first;
- `09_HGVSg_HGVSp_merge_summary.csv`: one summary row per processed sample.

If either HGVSg or HGVSp has duplicate merge keys, or if the two files do not contain the same exact variant/caller keys, the notebook saves an audit CSV and stops so the mismatch can be inspected.